In [1]:
import pandas as pd

df = pd.read_csv("./merged_price_news_2019_2024.csv")
df.head()


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Symbol,Capital Gains,Article,Url,Article_title
0,2019-03-28,75.911190,76.293664,75.260987,76.207603,1615600,0.0,0.0,A,NaN,Looking at the universe of stocks we cover at ...,https://www.nasdaq.com/articles/ex-dividend-re...,NaN
1,2019-04-17,74.878309,75.041190,71.342786,72.272179,4472000,0.0,0.0,A,NaN,Legendary investor Warren Buffett advises to b...,https://www.nasdaq.com/articles/agilent-techno...,NaN
2,2022-09-15,131.226510,132.893685,130.098719,130.589066,1446500,0.0,0.0,A,NaN,It's been said (and perhaps you are getting ti...,https://www.nasdaq.com/articles/agilent-a-down...,Agilent (A) Down 6.4% Since Last Earnings Repo...
3,2022-09-16,129.569112,129.578929,125.803256,127.382172,2300600,0.0,0.0,A,NaN,"Abbi Adest\nMistras Group ( MG ), a company wh...",https://www.nasdaq.com/articles/why-this-1-com...,Why This 1 Computer and Technology Stock Could...
4,2022-09-16,129.569112,129.578929,125.803256,127.382172,2300600,0.0,0.0,A,NaN,Rakesh Saxena AA\nSee also An Interview With t...,https://www.nasdaq.com/articles/first-week-of-...,First Week of May 2023 Options Trading For Agi...


In [2]:
df = df.sort_values("Date").reset_index(drop=True)
df

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Symbol,Capital Gains,Article,Url,Article_title
0,2019-01-02,29.820000,32.689999,29.290001,32.520000,11890300,0.0,0.0,ROKU,NaN,Stocks greeted the new year on Wednesday by wa...,https://www.nasdaq.com/articles/what-happened-...,What Happened in the Stock Market Today
1,2019-01-02,37.169442,37.302900,36.379474,36.555981,10549600,0.0,0.0,NEE,NaN,World markets have been in turmoil over the pa...,https://www.nasdaq.com/articles/3-top-energy-s...,3 Top Energy Stocks to Buy Right Now
2,2019-01-02,37.169442,37.302900,36.379474,36.555981,10549600,0.0,0.0,NEE,NaN,Looking today at week-over-week shares outstan...,https://www.nasdaq.com/articles/spyg-nee-adp-i...,"SPYG, NEE, ADP, ISRG: ETF Outflow Alert"
3,2019-01-02,136.730019,139.815384,134.020682,139.815384,82300,0.0,0.0,UNF,NaN,The following companies are expected to report...,https://www.nasdaq.com/articles/pre-market-ear...,"Pre-Market Earnings Report for January 3, 2019..."
4,2019-01-02,71.795730,76.231475,71.177894,75.423538,2385200,0.0,0.0,FANG,NaN,"Recently, The Carlyle Group LPCG completed the...",https://www.nasdaq.com/articles/carlyle-group-...,Carlyle Group Concludes Majority Stake Buyout ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455880,2024-12-31,519.880356,524.296604,516.722992,519.790405,1262900,0.0,0.0,TMO,NaN,Below is Validea's guru fundamental report for...,https://www.nasdaq.com/articles/tmo-quantitati...,NaN
1455881,2024-12-31,75.411600,75.904873,75.184691,75.431328,1133800,0.0,0.0,SYY,NaN,"Sysco Corporation SYY, a leader in the foodser...",https://www.nasdaq.com/articles/syscos-operati...,NaN
1455882,2024-12-31,108.500000,108.690002,106.589996,108.080002,1549000,0.0,0.0,TWLO,NaN,Here are three stocks with buy ranks and stron...,https://www.nasdaq.com/articles/best-growth-st...,NaN
1455883,2024-12-31,47.294506,47.960908,47.075691,47.185097,722100,0.0,0.0,CAKE,NaN,Growth stocks are attractive to many investors...,https://www.nasdaq.com/articles/cheesecake-fac...,NaN


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

# Extract only the Close prices and normalize
scaler = MinMaxScaler()
closes = scaler.fit_transform(df[["Close"]].values)  # shape=(N,1)

# Create sliding windows: past WINDOW days → next HORIZON days
WINDOW, HORIZON = 50, 3
X, Y = [], []
for i in range(len(closes) - WINDOW - HORIZON + 1):
    X.append(closes[i : i + WINDOW, 0])
    Y.append(closes[i + WINDOW : i + WINDOW + HORIZON, 0])
X = np.stack(X)  # (num_samples, WINDOW)
Y = np.stack(Y)  # (num_samples, HORIZON)

# Train/validation split
split = int(0.8 * len(X))
X_train, Y_train = X[:split], Y[:split]
X_val,   Y_val   = X[split:], Y[split:]

In [4]:
# 2. PyTorch Dataset & DataLoader
class PriceDataset(Dataset):
    def __init__(self, X, Y):
        # X shape: (samples, WINDOW), Y shape: (samples, HORIZON)
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)  # (B, WINDOW, 1)
        self.Y = torch.tensor(Y, dtype=torch.float32)               # (B, HORIZON)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


In [5]:
train_loader = DataLoader(PriceDataset(X_train, Y_train), batch_size=64, shuffle=True)
val_loader   = DataLoader(PriceDataset(X_val,   Y_val),   batch_size=64)

In [6]:
# 3a. LSTM Model
class LSTMForecast(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, horizon=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, horizon)
    def forward(self, x):
        # x: (batch, seq_len, 1)
        _, (h_n, _) = self.lstm(x)
        h_last = h_n[-1]               # (batch, hidden_size)
        return self.head(h_last)       # (batch, horizon)


In [7]:
# 3b. Simple RNN Model
class RNNForecast(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, horizon=3):
        super().__init__()
        self.rnn  = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, horizon)
    def forward(self, x):
        _, h_n = self.rnn(x)
        h_last = h_n[-1]
        return self.head(h_last)

In [8]:
# 3c. 1D CNN Model
class CNNForecast(nn.Module):
    def __init__(self, in_channels=1, num_filters=32, kernel_size=3, horizon=3):
        super().__init__()
        # Conv1d expects (batch, channels, seq_len)
        self.conv = nn.Conv1d(in_channels, num_filters, kernel_size, padding=kernel_size//2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(num_filters, horizon)
    def forward(self, x):
        # x: (batch, seq_len, 1) → (batch, 1, seq_len)
        x = x.transpose(1, 2)
        features = torch.relu(self.conv(x))       # (batch, num_filters, seq_len)
        pooled   = self.pool(features).squeeze(-1)  # (batch, num_filters)
        return self.head(pooled)

In [9]:
# 3d. Transformer Model
class TransformerForecast(nn.Module):
    def __init__(self, input_size=1, d_model=64, nhead=4, num_layers=2, horizon=3):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head    = nn.Linear(d_model, horizon)
    def forward(self, x):
        # x: (batch, seq_len, 1)
        x = self.input_proj(x)           # (batch, seq_len, d_model)
        encoded = self.encoder(x)        # (batch, seq_len, d_model)
        pooled  = encoded.mean(dim=1)    # Mean‐pool over time
        return self.head(pooled)

In [10]:
# 4. Training loop over models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
architectures = [
    LSTMForecast,
    RNNForecast,
    CNNForecast,
    TransformerForecast
]

for ModelClass in architectures:
    model = ModelClass().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.L1Loss()
    
    print(f"\n=== Training {ModelClass.__name__} ===")
    for epoch in range(1, 21):
        # --- train ---
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_mae = train_loss / len(train_loader.dataset)

        # --- validate ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_loss += criterion(model(xb), yb).item() * xb.size(0)
        val_mae = val_loss / len(val_loader.dataset)

        print(f"Epoch {epoch:02d} | Train MAE: {train_mae:.4f} | Val MAE: {val_mae:.4f}")


=== Training LSTMForecast ===
Epoch 01 | Train MAE: 0.0006 | Val MAE: 0.0001
Epoch 02 | Train MAE: 0.0003 | Val MAE: 0.0002
Epoch 03 | Train MAE: 0.0002 | Val MAE: 0.0001
Epoch 04 | Train MAE: 0.0002 | Val MAE: 0.0001
Epoch 05 | Train MAE: 0.0001 | Val MAE: 0.0001
Epoch 06 | Train MAE: 0.0001 | Val MAE: 0.0000
Epoch 07 | Train MAE: 0.0001 | Val MAE: 0.0001
Epoch 08 | Train MAE: 0.0001 | Val MAE: 0.0000
Epoch 09 | Train MAE: 0.0001 | Val MAE: 0.0001
Epoch 10 | Train MAE: 0.0001 | Val MAE: 0.0001
Epoch 11 | Train MAE: 0.0001 | Val MAE: 0.0001
Epoch 12 | Train MAE: 0.0001 | Val MAE: 0.0002
Epoch 13 | Train MAE: 0.0001 | Val MAE: 0.0001
Epoch 14 | Train MAE: 0.0001 | Val MAE: 0.0002
Epoch 15 | Train MAE: 0.0001 | Val MAE: 0.0001
Epoch 16 | Train MAE: 0.0001 | Val MAE: 0.0001
Epoch 17 | Train MAE: 0.0001 | Val MAE: 0.0001
Epoch 18 | Train MAE: 0.0001 | Val MAE: 0.0001
Epoch 19 | Train MAE: 0.0001 | Val MAE: 0.0001
Epoch 20 | Train MAE: 0.0001 | Val MAE: 0.0001

=== Training RNNForecast ===